<a href="https://colab.research.google.com/github/Jasonmiller513/Dissertation/blob/main/corrected_one_step_qlearning_sarsa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Corrected Q-learning and SARSA: one-step fulfillment decisions

**Separate supplementary experiment; do not overwrite the original dissertation notebook.**

This notebook corrects the action/reward attribution, initializes **independent** Q-learning and conventional SARSA tables, uses the same training/evaluation data and hyperparameters, and defines **one historical order as one terminal episode**. Because the data do not support action-dependent future states, `gamma = 0`. This is a **contextual bandit / one-step RL benchmark**, not a validated sequential MDP. No claim of causal policy improvement follows from modeled outcomes.

**Action scope:** shipping mode on the *observed destination route*. The dataset does not verify alternative fulfillment origins or inventory availability. The original fulfillment-region/route/mode action space cannot be responsibly simulated without a separately validated operational model.

**Run order:** Run all cells in this notebook. It requires `dataco_rl_train.csv`, `dataco_rl_validation.csv`, and `dataco_rl_test.csv` in the current working directory. All outcome estimation is trained on train only. Validation is used for methodological choices; test is reported once after freezing them.

## 1. Load original temporal splits and construct pre-decision states

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, brier_score_loss

SEED = 42
rng = np.random.default_rng(SEED)
ROOT = Path('.')
CAT='Product Category Id'; REGION='Order Region'; COUNTRY='Order Country'
MODE='Shipping Mode'; PROFIT='Order Profit Per Order'
DAYS='Days for shipping (real)'; LATE='Late_delivery_risk'
QTY='Order Item Quantity'; VALUE='Order Item Total'
FILES={'train':'dataco_rl_train.csv','validation':'dataco_rl_validation.csv','test':'dataco_rl_test.csv'}
frames={k:pd.read_csv(ROOT/v,encoding='latin1') for k,v in FILES.items()}
needed=[CAT,REGION,COUNTRY,MODE,PROFIT,DAYS,LATE,QTY,VALUE]
for name,d in frames.items():
    missing=[c for c in needed if c not in d]
    if missing: raise KeyError(f'{name}: missing {missing}')
    for c in [CAT,REGION,COUNTRY,MODE]: d[c]=d[c].fillna('Unknown').astype(str)
    for c in [PROFIT,DAYS,LATE,QTY,VALUE]: d[c]=pd.to_numeric(d[c],errors='coerce')
    d['Route']=d[REGION]+' -> '+d[COUNTRY]
    d['valid_target']=d[[PROFIT,DAYS,LATE]].notna().all(axis=1)&d[DAYS].ge(0)&d[LATE].between(0,1)
train=frames['train'].loc[frames['train'].valid_target].copy().reset_index(drop=True)
val=frames['validation'].copy().reset_index(drop=True)
test=frames['test'].copy().reset_index(drop=True)
print({k:len(v) for k,v in frames.items()})

{'train': 126363, 'validation': 27078, 'test': 27078}


## 2. Train-only state bins and historical outcome model

In [2]:
# State includes ONLY information available before selecting a shipping mode.
# Train-only numeric bin boundaries and missing-value imputation.
FILL={c:float(train[c].median()) if train[c].notna().any() else 0.0 for c in [QTY,VALUE]}
EDGES={c:np.unique(np.quantile(train[c].fillna(FILL[c]),np.linspace(0,1,5)))[1:-1] for c in [QTY,VALUE]}

def make_states(d):
    d=d.copy()
    for c in [QTY,VALUE]:
        d[c+'_bin']=np.searchsorted(EDGES[c],d[c].fillna(FILL[c]).to_numpy(),side='right')
    # Do not include selected shipping mode, realized profit, shipping delay, late risk, or realized cost.
    d['state']=d[[CAT,REGION,COUNTRY,QTY+'_bin',VALUE+'_bin']].astype(str).agg('|'.join,axis=1)
    return d
train,val,test=[make_states(x) for x in (train,val,test)]

MIN_SUPPORT=10
LEVELS=[([CAT,'Route',MODE],'category_route_mode'),(['Route',MODE],'route_mode'),([REGION,MODE],'region_mode'),([MODE],'mode')]
class StateAwareEstimator:
    def fit(self,d):
        self.fills=FILL
        z=d.copy()
        for c in [QTY,VALUE]: z[c]=z[c].fillna(self.fills[c])
        means=z.groupby(LEVELS[0][0])[[PROFIT,QTY,VALUE]].transform('mean')
        X=(z[[QTY,VALUE]]-means[[QTY,VALUE]]).to_numpy(float)
        y=(z[PROFIT]-means[PROFIT]).to_numpy(float)
        self.scale=np.where(X.std(axis=0)>1e-12,X.std(axis=0),1.)
        self.beta=Ridge(alpha=100.,fit_intercept=False).fit(X/self.scale,y).coef_/self.scale
        self.tables=[]
        for keys,label in LEVELS:
            tbl=z.groupby(keys,dropna=False).agg(n=(PROFIT,'size'),profit=(PROFIT,'mean'),days=(DAYS,'mean'),late=(LATE,'mean'),mean_qty=(QTY,'mean'),mean_value=(VALUE,'mean')).reset_index()
            self.tables.append((keys,label,tbl))
        return self
    def predict(self,d):
        out=d.reset_index(drop=True).copy()
        for c in [QTY,VALUE]:out[c]=out[c].fillna(self.fills[c])
        for c in ['profit','days','late','support']:out['estimated_'+c]=np.nan
        out['estimated_level']='unsupported'
        for keys,label,tbl in self.tables:
            m=out[keys].merge(tbl,on=keys,how='left',sort=False)
            use=out['estimated_level'].eq('unsupported').to_numpy()&m['n'].ge(MIN_SUPPORT).fillna(False).to_numpy()
            if not use.any():continue
            adjustment=self.beta[0]*(out.loc[use,QTY].to_numpy()-m.loc[use,'mean_qty'].to_numpy())+self.beta[1]*(out.loc[use,VALUE].to_numpy()-m.loc[use,'mean_value'].to_numpy())
            out.loc[use,'estimated_profit']=m.loc[use,'profit'].to_numpy()+adjustment
            out.loc[use,'estimated_days']=m.loc[use,'days'].to_numpy()
            out.loc[use,'estimated_late']=m.loc[use,'late'].to_numpy()
            out.loc[use,'estimated_support']=m.loc[use,'n'].to_numpy()
            out.loc[use,'estimated_level']=label
        return out

estimator=StateAwareEstimator().fit(train)
print('Training-only profit coefficients:',dict(zip([QTY,VALUE],estimator.beta)))

Training-only profit coefficients: {'Order Item Quantity': np.float64(0.24218478866263862), 'Order Item Total': np.float64(0.10805209547936388)}


## 3. Action eligibility, modeled rewards, and counterfactual support

In [3]:
# Limit alternative shipping modes to those historically observed on the SAME route.
# Do not claim alternative fulfillment origins are verified.
route_modes=train.groupby('Route')[MODE].unique().apply(lambda x:sorted(set(x))).to_dict()
MODES=sorted(train[MODE].unique())
MODE_TO_ID={m:i for i,m in enumerate(MODES)}
STRICT={'category_route_mode','route_mode'}
# Scenario coefficients: not empirically measured causal effects. Freeze after validation.
W_DAYS=0.5; W_LATE=2.0

def candidate_table(d):
    chunks=[]
    # Vectorized over orders, once per mode, not per order.
    for mode in MODES:
        ix=np.flatnonzero(d['Route'].map(lambda route:mode in route_modes.get(route,[])).to_numpy())
        if not len(ix):continue
        alt=d.iloc[ix].copy().reset_index(drop=True)
        alt[MODE]=mode
        pred=estimator.predict(alt)
        valid=pred['estimated_level'].isin(STRICT)&pred[['estimated_profit','estimated_days','estimated_late']].notna().all(axis=1)
        if not valid.any():continue
        part=pd.DataFrame({'row_id':ix[valid.to_numpy()],'state':alt.loc[valid,'state'].to_numpy(),
            'mode':mode,'action_id':MODE_TO_ID[mode],
            'profit':pred.loc[valid,'estimated_profit'].to_numpy(),
            'days':pred.loc[valid,'estimated_days'].to_numpy(),
            'late':pred.loc[valid,'estimated_late'].to_numpy(),
            'support':pred.loc[valid,'estimated_support'].to_numpy(),
            'match_level':pred.loc[valid,'estimated_level'].to_numpy()})
        part['reward']=part['profit']-W_DAYS*part['days']-W_LATE*part['late']
        chunks.append(part)
    return pd.concat(chunks,ignore_index=True) if chunks else pd.DataFrame(columns=['row_id','state','mode','action_id','profit','days','late','support','match_level','reward'])

train_candidates=candidate_table(train)
val_candidates=candidate_table(val)
print('Training orders with supported candidate:',train_candidates.row_id.nunique(),'/',len(train))
print('Validation orders with supported candidate:',val_candidates.row_id.nunique(),'/',len(val))
print('Candidate matching levels:',train_candidates.match_level.value_counts().to_dict())

Training orders with supported candidate: 126193 / 126363
Validation orders with supported candidate: 27028 / 27078
Candidate matching levels: {'category_route_mode': 406928, 'route_mode': 90441}


## 4. Independent Q-learning and conventional SARSA with terminal one-step episodes

For each historical order, the learner selects a shipping mode from that order's supported alternatives, receives its **estimated action-specific** reward, updates the selected state-action value, and **terminates**. No subsequent historical order is treated as a consequence of the selected action.

Terminal Q-learning target: `r + gamma*0 = r`. Terminal SARSA target: `r + gamma*0 = r`. Therefore the updates are mathematically identical when exposed to identical samples. Any empirical difference reflects stochastic exploration/training variation, **not sequential RL superiority**.

In [4]:
# Encode only states seen in training. Unknown validation/test states are explicitly unsupported.
STATES=sorted(train_candidates['state'].unique())
STATE_TO_ID={s:i for i,s in enumerate(STATES)}
train_candidates['state_id']=train_candidates['state'].map(STATE_TO_ID).astype(int)

# One candidate reward for every eligible (order,mode); train episodes sample orders.
by_order={int(i):g[['state_id','action_id','reward']].to_numpy(float) for i,g in train_candidates.groupby('row_id')}
order_ids=np.array(sorted(by_order),dtype=int)
n_states=len(STATES); n_actions=len(MODES)

# Identical hyperparameters; separate Q-tables and independent RNG streams.
EPISODES=20   # one epoch = one pass through all supported orders; tune using validation only
ALPHA=0.10
EPS_START=0.30; EPS_END=0.03
GAMMA=0.0
assert GAMMA==0.0,'Action-independent historical transitions cannot justify future-state bootstrapping.'

def train_one_step(algorithm,seed):
    assert algorithm in ('Q-learning','SARSA')
    local=np.random.default_rng(seed)
    Q=np.zeros((n_states,n_actions),dtype=float)
    visits=np.zeros_like(Q,dtype=int)
    logs=[]
    for epoch in range(EPISODES):
        epsilon=EPS_END+(EPS_START-EPS_END)*(1-epoch/max(EPISODES-1,1))
        errors=[]
        for order in local.permutation(order_ids):
            choices=by_order[int(order)]
            s=int(choices[0,0]); actions=choices[:,1].astype(int)
            if local.random()<epsilon:a=int(local.choice(actions))
            else:a=int(actions[np.argmax(Q[s,actions])])
            r=float(choices[np.flatnonzero(actions==a)[0],2])
            # One-step terminal episode. No future-state value.
            terminal=True
            if algorithm=='Q-learning':
                target=r if terminal else np.nan  # future max-Q is inapplicable
            else:
                target=r if terminal else np.nan  # no next action at terminal
            td=target-Q[s,a]
            Q[s,a]+=ALPHA*td
            visits[s,a]+=1
            errors.append(abs(td))
        logs.append({'algorithm':algorithm,'epoch':epoch+1,'mean_abs_td_error':np.mean(errors),'epsilon':epsilon})
    return Q,visits,pd.DataFrame(logs)

Q_ql,visits_ql,log_ql=train_one_step('Q-learning',SEED)
Q_sarsa,visits_sarsa,log_sarsa=train_one_step('SARSA',SEED+1)
print('Q-learning and SARSA independently trained; gamma=',GAMMA)
print('State/action table shape:',Q_ql.shape)
print(pd.concat([log_ql.tail(1),log_sarsa.tail(1)]).to_string(index=False))

Q-learning and SARSA independently trained; gamma= 0.0
State/action table shape: (6167, 4)
 algorithm  epoch  mean_abs_td_error  epsilon
Q-learning     20           1.939754     0.03
     SARSA     20           1.949102     0.03


## 5. Fair validation evaluation (same modeled scale for observed and recommended actions)

In [5]:
def evaluate_policy(d,candidates,Q,label):
    z=candidates.copy()
    z['state_id']=z['state'].map(STATE_TO_ID)
    z=z[z.state_id.notna()].copy()
    z['state_id']=z['state_id'].astype(int)
    if z.empty:
        print(label,'No eligible previously seen states');return pd.DataFrame(),{}
    # Select highest learned Q among eligible actions for each order.
    z['q_value']=Q[z.state_id.to_numpy(),z.action_id.to_numpy()]
    chosen=z.sort_values(['row_id','q_value','action_id'],ascending=[True,False,True]).drop_duplicates('row_id')
    # Modeled historical action using the SAME estimator, same eligible sample.
    obs=z[z['mode'].to_numpy()==d.iloc[z.row_id.to_numpy()][MODE].to_numpy()].copy()
    obs=obs.drop_duplicates('row_id')
    paired=chosen.merge(obs[['row_id','profit','days','late','reward']],on='row_id',how='inner',suffixes=('_policy','_historical'))
    metrics={'algorithm':label,'eligible_orders':len(paired),'total_orders':len(d),'coverage':len(paired)/len(d)}
    for m in ['profit','days','late','reward']:
        metrics['modeled_historical_'+m]=paired[m+'_historical'].mean() if len(paired) else np.nan
        metrics['modeled_policy_'+m]=paired[m+'_policy'].mean() if len(paired) else np.nan
    print(label,'paired supported orders:',len(paired),'/',len(d))
    return paired,metrics

val_ql,metrics_val_ql=evaluate_policy(val,val_candidates,Q_ql,'Q-learning')
val_sarsa,metrics_val_sarsa=evaluate_policy(val,val_candidates,Q_sarsa,'SARSA')
validation_summary=pd.DataFrame([metrics_val_ql,metrics_val_sarsa])
display(validation_summary)
print('NOTE: These are modeled comparisons, not causal estimates of real policy gains.')

Q-learning paired supported orders: 25886 / 27078
SARSA paired supported orders: 25886 / 27078


,algorithm,eligible_orders,total_orders,coverage,modeled_historical_profit,modeled_policy_profit,modeled_historical_days,modeled_policy_days,modeled_historical_late,modeled_policy_late,modeled_historical_reward,modeled_policy_reward
0,Q-learning,25886,27078,0.955979,21.711637,35.202402,3.511677,1.926632,0.541906,0.625397,18.871986,32.988292
1,SARSA,25886,27078,0.955979,21.711637,35.196922,3.511677,1.953517,0.541906,0.628833,18.871986,32.962498


NOTE: These are modeled comparisons, not causal estimates of real policy gains.


In [6]:
assert {'train','validation','test'} <= set(frames)
assert len(val)==len(frames['validation'])
print('Validation rows:',len(val),'Test rows:',len(test))
print('Previously reported table (validation):')
display(validation_summary)
print('Validation summary was created by evaluate_policy(val, val_candidates, ...).')
print('Final test is evaluated separately by cell 12 of the original corrected notebook.')

Validation rows: 27078 Test rows: 27078
Previously reported table (validation):


,algorithm,eligible_orders,total_orders,coverage,modeled_historical_profit,modeled_policy_profit,modeled_historical_days,modeled_policy_days,modeled_historical_late,modeled_policy_late,modeled_historical_reward,modeled_policy_reward
0,Q-learning,25886,27078,0.955979,21.711637,35.202402,3.511677,1.926632,0.541906,0.625397,18.871986,32.988292
1,SARSA,25886,27078,0.955979,21.711637,35.196922,3.511677,1.953517,0.541906,0.628833,18.871986,32.962498


Validation summary was created by evaluate_policy(val, val_candidates, ...).
Final test is evaluated separately by cell 12 of the original corrected notebook.


In [7]:
import numpy as np, pandas as pd

# Use existing train_candidates, val_candidates, STATES, STATE_TO_ID, MODE_TO_ID,
# n_states, n_actions, ALPHA, EPISODES, EPS_START, EPS_END, MODE, and train.
# All outcomes below are already estimated by the TRAIN-fitted estimator.

def add_reward(df, days_weight, late_weight):
    out=df.copy()
    out['reward']=out['profit']-days_weight*out['days']-late_weight*out['late']
    return out

def train_repeated(candidates, algorithm, seed, epochs=EPISODES):
    assert algorithm in ('Q-learning','SARSA')
    rng=np.random.default_rng(seed)
    Q=np.zeros((n_states,n_actions),dtype=float)
    z=candidates.copy()
    z['state_id']=z['state'].map(STATE_TO_ID)
    z=z[z['state_id'].notna()].copy()
    z['state_id']=z['state_id'].astype(int)
    by_order={int(i):g[['state_id','action_id','reward']].to_numpy(float)
              for i,g in z.groupby('row_id',sort=False)}
    ids=np.array(sorted(by_order),dtype=int)
    for epoch in range(epochs):
        eps=EPS_END+(EPS_START-EPS_END)*(1-epoch/max(epochs-1,1))
        for order_id in rng.permutation(ids):
            choices=by_order[int(order_id)]
            s=int(choices[0,0]); actions=choices[:,1].astype(int)
            if rng.random()<eps:
                a=int(rng.choice(actions))
            else:
                a=int(actions[np.argmax(Q[s,actions])])
            r=float(choices[np.flatnonzero(actions==a)[0],2])
            # Both updates reduce to immediate reward because every step is terminal.
            Q[s,a]+=ALPHA*(r-Q[s,a])
    return Q

def paired_for(Q,candidates,orders):
    # Identical eligibility and matched historical-action benchmark for every policy.
    z=candidates.copy()
    z['state_id']=z['state'].map(STATE_TO_ID)
    z=z[z.state_id.notna()].copy()
    z['state_id']=z['state_id'].astype(int)
    z['q']=Q[z.state_id.to_numpy(),z.action_id.to_numpy()]
    chosen=(z.sort_values(['row_id','q','action_id'],ascending=[True,False,True])
             .drop_duplicates('row_id'))
    recorded=orders.iloc[z.row_id.to_numpy()][MODE].to_numpy()
    observed=z[z['mode'].to_numpy()==recorded].drop_duplicates('row_id')
    return chosen[['row_id','mode','profit','days','late','reward']].merge(
        observed[['row_id','profit','days','late','reward']],on='row_id',
        how='inner',suffixes=('_policy','_historical')).sort_values('row_id').reset_index(drop=True)

def summarize_paired(p,algorithm,seed,wd,wl):
    d={'algorithm':algorithm,'seed':seed,'days_weight':wd,'late_weight':wl,
       'eligible_orders':len(p),'coverage':len(p)/len(val)}
    for m in ['profit','days','late','reward']:
        d['historical_'+m]=p[m+'_historical'].mean()
        d['policy_'+m]=p[m+'_policy'].mean()
        d['delta_'+m]=d['policy_'+m]-d['historical_'+m]
    return d

In [8]:
def bootstrap_difference(q_pairs,s_pairs,metric,draws=2000,seed=2026):
    q=q_pairs[['row_id',metric+'_policy']].rename(columns={metric+'_policy':'q'})
    s=s_pairs[['row_id',metric+'_policy']].rename(columns={metric+'_policy':'s'})
    joined=q.merge(s,on='row_id',how='inner',validate='one_to_one')
    diff=(joined['q']-joined['s']).to_numpy(float)
    if len(diff)<2: raise ValueError('Insufficient paired orders')
    rng=np.random.default_rng(seed)
    # Memory-efficient bootstrap (do not allocate draws x number_of_orders).
    means=np.empty(draws)
    for j in range(draws):
        means[j]=diff[rng.integers(0,len(diff),len(diff))].mean()
    lo,hi=np.percentile(means,[2.5,97.5])
    return {'metric':metric,'paired_orders':len(diff),'q_minus_s':diff.mean(),
            'ci_95_low':lo,'ci_95_high':hi,
            'ci_excludes_zero':bool(lo>0 or hi<0)}

paired_ci=pd.DataFrame([bootstrap_difference(val_ql,val_sarsa,m) for m in ['profit','days','late','reward']])
display(paired_ci)
print('For profit/reward, positive Q−SARSA favors Q. For days/late, negative favors Q.')

,metric,paired_orders,q_minus_s,ci_95_low,ci_95_high,ci_excludes_zero
0,profit,25886,0.005479,-0.023698,0.033563,False
1,days,25886,-0.026886,-0.034152,-0.019384,True
2,late,25886,-0.003436,-0.004591,-0.002356,True
3,reward,25886,0.025793,-0.003558,0.054822,False


For profit/reward, positive Q−SARSA favors Q. For days/late, negative favors Q.


In [9]:
# Adjust these only based on validation and record any change in the dissertation.
WEIGHT_SCENARIOS=[
    ('original',0.5,2.0),
    ('moderate_late',0.5,20.0),
    ('strong_late',0.5,50.0),
    ('very_strong_late',0.5,100.0),
]
SEEDS=[42,101,202,303,404]
records=[]
for scenario,wd,wl in WEIGHT_SCENARIOS:
    tc=add_reward(train_candidates,wd,wl)
    vc=add_reward(val_candidates,wd,wl)
    for seed in SEEDS:
        for alg,offset in [('Q-learning',0),('SARSA',100000)]:
            Q=train_repeated(tc,alg,seed+offset)
            paired=paired_for(Q,vc,val)
            rec=summarize_paired(paired,alg,seed,wd,wl)
            rec['scenario']=scenario
            records.append(rec)
            print(scenario,alg,'seed',seed,'n=',len(paired),
                  'Δprofit=',round(rec['delta_profit'],3),
                  'Δdays=',round(rec['delta_days'],3),
                  'Δlate=',round(rec['delta_late'],4))
seed_results=pd.DataFrame(records)
seed_results.to_csv('corrected_validation_seed_weight_results.csv',index=False)
summary=(seed_results.groupby(['scenario','algorithm','days_weight','late_weight'])
         .agg(n_seeds=('seed','nunique'),orders=('eligible_orders','min'),
              mean_delta_profit=('delta_profit','mean'),sd_delta_profit=('delta_profit','std'),
              mean_delta_days=('delta_days','mean'),sd_delta_days=('delta_days','std'),
              mean_delta_late=('delta_late','mean'),sd_delta_late=('delta_late','std'),
              mean_delta_reward=('delta_reward','mean')).reset_index())
summary['all_three_improve']=(summary.mean_delta_profit>0)&(summary.mean_delta_days<0)&(summary.mean_delta_late<0)
display(summary)
summary.to_csv('corrected_validation_weight_sensitivity.csv',index=False)

original Q-learning seed 42 n= 25886 Δprofit= 13.491 Δdays= -1.585 Δlate= 0.0835
original SARSA seed 42 n= 25886 Δprofit= 13.423 Δdays= -1.569 Δlate= 0.0847
original Q-learning seed 101 n= 25886 Δprofit= 13.491 Δdays= -1.572 Δlate= 0.0822
original SARSA seed 101 n= 25886 Δprofit= 13.439 Δdays= -1.574 Δlate= 0.0836
original Q-learning seed 202 n= 25886 Δprofit= 13.501 Δdays= -1.545 Δlate= 0.0848
original SARSA seed 202 n= 25886 Δprofit= 13.504 Δdays= -1.552 Δlate= 0.0868
original Q-learning seed 303 n= 25886 Δprofit= 13.463 Δdays= -1.55 Δlate= 0.0881
original SARSA seed 303 n= 25886 Δprofit= 13.466 Δdays= -1.557 Δlate= 0.0877
original Q-learning seed 404 n= 25886 Δprofit= 13.463 Δdays= -1.574 Δlate= 0.0841
original SARSA seed 404 n= 25886 Δprofit= 13.421 Δdays= -1.563 Δlate= 0.0861
moderate_late Q-learning seed 42 n= 25886 Δprofit= 12.921 Δdays= -1.526 Δlate= -0.0212
moderate_late SARSA seed 42 n= 25886 Δprofit= 12.895 Δdays= -1.512 Δlate= -0.022
moderate_late Q-learning seed 101 n= 258

,scenario,algorithm,days_weight,late_weight,n_seeds,orders,mean_delta_profit,sd_delta_profit,mean_delta_days,sd_delta_days,mean_delta_late,sd_delta_late,mean_delta_reward,all_three_improve
0,moderate_late,Q-learning,0.5,20.0,5,25886,12.874830,0.040601,-1.525855,0.006080,-0.023322,0.001870,14.104192,True
1,moderate_late,SARSA,0.5,20.0,5,25886,12.845681,0.046419,-1.521105,0.012411,-0.024564,0.001829,14.097506,True
2,original,Q-learning,0.5,2.0,5,25886,13.481698,0.017433,-1.564948,0.017046,0.084516,0.002194,14.095141,False
3,original,SARSA,0.5,2.0,5,25886,13.450470,0.034970,-1.563016,0.008870,0.085761,0.001621,14.060455,False
4,strong_late,Q-learning,0.5,50.0,5,25886,9.940344,0.064555,-1.230827,0.019246,-0.119000,0.001504,16.505735,True
5,strong_late,SARSA,0.5,50.0,5,25886,9.955506,0.037094,-1.217995,0.006182,-0.118819,0.000729,16.505449,True
6,very_strong_late,Q-learning,0.5,100.0,5,25886,8.036974,0.057676,-0.973702,0.010405,-0.148245,0.000674,23.348341,True
7,very_strong_late,SARSA,0.5,100.0,5,25886,8.018583,0.028642,-0.972377,0.006530,-0.148296,0.000321,23.334339,True


In [10]:
# Final selected reward configuration
# Based on validation results only

days_weight = 0.5
late_weight = 50.0
SEED = 42

print("Selected late penalty:", late_weight)
print("Selected delivery penalty:", days_weight)
print("Selected training seed:", SEED)

Selected late penalty: 50.0
Selected delivery penalty: 0.5
Selected training seed: 42


In [11]:
# FINAL CONFIGURATION — selected using validation only
FINAL_DAYS_WEIGHT = 0.5
FINAL_LATE_WEIGHT = 50.0
FINAL_SEED = 42

# Recalculate rewards using the selected configuration
final_train_candidates = add_reward(
    train_candidates,
    FINAL_DAYS_WEIGHT,
    FINAL_LATE_WEIGHT
)

# Train independent models using training data only
Q_ql_final = train_repeated(
    final_train_candidates,
    "Q-learning",
    FINAL_SEED
)

Q_sarsa_final = train_repeated(
    final_train_candidates,
    "SARSA",
    FINAL_SEED + 100000
)

print("Final Q-learning and SARSA training complete.")
print("Selected weights:", FINAL_DAYS_WEIGHT, FINAL_LATE_WEIGHT)
print("Q-table shape:", Q_ql_final.shape)
print("SARSA table shape:", Q_sarsa_final.shape)

Final Q-learning and SARSA training complete.
Selected weights: 0.5 50.0
Q-table shape: (6167, 4)
SARSA table shape: (6167, 4)


In [12]:
# FINAL TEST EVALUATION
# Do not change model settings based on these results.

# Generate test candidates using the training-fitted estimator
test_candidates_final = candidate_table(test)

# Apply the selected reward weights
test_candidates_final = add_reward(
    test_candidates_final,
    FINAL_DAYS_WEIGHT,
    FINAL_LATE_WEIGHT
)

# Evaluate independently trained policies
test_ql_final = paired_for(
    Q_ql_final,
    test_candidates_final,
    test
)

test_sarsa_final = paired_for(
    Q_sarsa_final,
    test_candidates_final,
    test
)

# Summarize test performance
def final_test_summary(paired, algorithm):
    results = {
        "algorithm": algorithm,
        "eligible_orders": len(paired),
        "total_orders": len(test),
        "coverage": len(paired) / len(test)
    }

    for metric in ["profit", "days", "late", "reward"]:
        historical = paired[f"{metric}_historical"].mean()
        policy = paired[f"{metric}_policy"].mean()

        results[f"historical_{metric}"] = historical
        results[f"policy_{metric}"] = policy
        results[f"delta_{metric}"] = policy - historical

    return results

final_results = pd.DataFrame([
    final_test_summary(test_ql_final, "Q-learning"),
    final_test_summary(test_sarsa_final, "SARSA")
])

display(final_results)

final_results.to_csv(
    "corrected_final_test_results.csv",
    index=False
)

print("Final test results saved.")

,algorithm,eligible_orders,total_orders,coverage,historical_profit,policy_profit,delta_profit,historical_days,policy_days,delta_days,historical_late,policy_late,delta_late,historical_reward,policy_reward,delta_reward
0,Q-learning,17235,27078,0.636495,22.88335,32.840332,9.956982,3.495941,1.822902,-1.673040,0.551466,0.434524,-0.116942,-6.437924,10.202669,16.640593
1,SARSA,17235,27078,0.636495,22.88335,32.197959,9.314609,3.495941,1.879302,-1.616639,0.551466,0.421560,-0.129906,-6.437924,10.180294,16.618218


Final test results saved.


In [13]:
# FINAL GOAL ASSESSMENT

for _, row in final_results.iterrows():

    profit_pct = (
        (row["policy_profit"] - row["historical_profit"])
        / abs(row["historical_profit"]) * 100
    )

    days_pct = (
        (row["historical_days"] - row["policy_days"])
        / row["historical_days"] * 100
    )

    historical_on_time = 1 - row["historical_late"]
    policy_on_time = 1 - row["policy_late"]

    on_time_pct = (
        (policy_on_time - historical_on_time)
        / historical_on_time * 100
    )

    print("\nAlgorithm:", row["algorithm"])
    print(f"Modeled profit improvement: {profit_pct:.2f}%")
    print(f"Modeled delivery-time reduction: {days_pct:.2f}%")
    print(f"Modeled on-time improvement: {on_time_pct:.2f}%")

    goal_met = (
        profit_pct >= 10
        and on_time_pct >= 20
    )

    print("Modeled profit/on-time thresholds met:", goal_met)


Algorithm: Q-learning
Modeled profit improvement: 43.51%
Modeled delivery-time reduction: 47.86%
Modeled on-time improvement: 26.07%
Modeled profit/on-time thresholds met: True

Algorithm: SARSA
Modeled profit improvement: 40.70%
Modeled delivery-time reduction: 46.24%
Modeled on-time improvement: 28.96%
Modeled profit/on-time thresholds met: True


## 6. Freeze design; evaluate untouched test once

In [ ]:
# Run this cell only after deciding all parameters, eligibility, and estimator choices on validation.
# It does not retrain or tune on test.
test_candidates=candidate_table(test)
test_ql,metrics_test_ql=evaluate_policy(test,test_candidates,Q_ql,'Q-learning')
test_sarsa,metrics_test_sarsa=evaluate_policy(test,test_candidates,Q_sarsa,'SARSA')
test_summary=pd.DataFrame([metrics_test_ql,metrics_test_sarsa])
display(test_summary)
test_summary.to_csv('corrected_one_step_algorithm_comparison.csv',index=False)
validation_summary.to_csv('corrected_one_step_validation_comparison.csv',index=False)
print('Saved corrected_one_step_algorithm_comparison.csv and corrected_one_step_validation_comparison.csv')

Q-learning paired supported orders: 17235 / 27078
SARSA paired supported orders: 17235 / 27078


,algorithm,eligible_orders,total_orders,coverage,modeled_historical_profit,modeled_policy_profit,modeled_historical_days,modeled_policy_days,modeled_historical_late,modeled_policy_late,modeled_historical_reward,modeled_policy_reward
0,Q-learning,17235,27078,0.636495,22.88335,36.575099,3.495941,2.050638,0.551466,0.671719,20.032447,34.206343
1,SARSA,17235,27078,0.636495,22.88335,36.530753,3.495941,1.957792,0.551466,0.675950,20.032447,34.199956


Saved corrected_one_step_algorithm_comparison.csv and corrected_one_step_validation_comparison.csv


## 7. Dissertation interpretation and limitations

- **Decision step:** one historical order and one eligible shipping-mode choice on its observed destination route.
- **Episode:** exactly one decision and immediate modeled reward. The order terminates immediately.
- **Transition:** terminal; there is no `s_{t+1}` and no action-dependent evolution of inventory or capacity.
- **Termination rule:** after evaluating the selected action once.
- **Cumulative rewards:** none are used; `gamma=0`.
- **Outcome model:** train-only hierarchical matching with minimum support of 10 and state-aware profit adjustment. Estimates of unobserved actions are not causally verified.
- **Algorithm comparison:** Q-learning and conventional SARSA are implemented independently but become equivalent terminal TD updates; observed differences are sampling/optimization variation and cannot demonstrate an advantage of sequential off-policy over on-policy control.
- **Scope limitation:** does not evaluate changing fulfillment origin. Candidate fulfillment regions in the original notebook were inferred and did not establish physical inventory availability.
- **Policy evaluation:** only held-out orders with supported observed and recommended actions and states seen during training; coverage must be reported. No observed-profit-based postselection.

**If the dissertation requires a genuine sequential Q-learning versus SARSA comparison**, construct and validate an operational simulator with inventory, replenishment, capacities, and action-dependent transitions, then rerun both algorithms. Do not relabel this one-step benchmark as such a simulator.